# Minimum Sample Size Calculation

**Method:** Frequentist z-test for a single proportion (normal approximation to the binomial)

---

$$N = \frac{[z_{\alpha}\sqrt{p_0(1-p_0)} + z_{\beta}\sqrt{\hat{p}(1-\hat{p})}]^2}{(\hat{p} - p_0)^2}$$

---

| Symbol | Meaning |
|--------|---------|
| $N$ | Minimum sample size (visitors) needed |
| $z_{\alpha}$ | z-score for desired confidence level (Type I error) |
| $z_{\beta}$ | z-score for desired statistical power (Type II error) |
| $\hat{p}$ | Expected (true) conversion rate |
| $p_0$ | Target benchmark conversion rate |

## Data Input

Test design parameters are loaded from `test_config.json`. Edit the config file to change test parameters.

In [28]:
import json

config = json.load(open("../test_config.json"))
d = config["design"]

# ==========================================
#  TEST DESIGN (from test_config.json)
# ==========================================
#  target_rate   : 0.02   — benchmark CVR (p₀)
#  expected_rate : 0.01   — expected true CVR (p̂)
#  alpha         : 0.05   — significance level
#  power         : 0.80   — statistical power
#  two_sided     : true   — test type
#  n_variants    : 1      — number of variants
# ==========================================
print(f"Design: target={d['target_rate']:.1%}, expected={d['expected_rate']:.1%}, "
      f"α={d['alpha']}, power={d['power']:.0%}, "
      f"{'two-sided' if d['two_sided'] else 'one-sided'}, {d['n_variants']} variant(s)")

Design: target=2.0%, expected=1.0%, α=0.05, power=80%, two-sided, 1 variant(s)


In [29]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import norm

def calculate_min_sample_size(target_rate, expected_rate, two_sided=False, alpha=0.05, power=0.80, n_variants=1):
    alpha_adj = alpha / n_variants

    if expected_rate == target_rate:
        print(f"ERROR: expected_rate ({expected_rate:.1%}) cannot equal target ({target_rate:.1%})")
        return
    if not two_sided and expected_rate <= target_rate:
        print(f"ERROR: For one-sided test, expected_rate ({expected_rate:.1%}) must be > target ({target_rate:.1%})")
        return

    z_alpha = norm.ppf(1 - alpha_adj / 2) if two_sided else norm.ppf(1 - alpha_adj)
    z_beta = norm.ppf(power)
    test_label = "Two-Sided" if two_sided else "One-Sided"

    var_expected = expected_rate * (1 - expected_rate)
    var_target = target_rate * (1 - target_rate)
    effect_sq = (expected_rate - target_rate) ** 2
    numerator = (z_alpha * np.sqrt(var_target) + z_beta * np.sqrt(var_expected)) ** 2
    required_n = int(np.ceil(numerator / effect_sq))

    # --- Output ---
    print(f"--- SAMPLE SIZE PLANNER ({test_label}) ---")
    print(f"手法 : 頻度論 z検定（単一比率の検定）")
    print(f"目標 : {target_rate:.1%}  |  期待CVR : {expected_rate:.1%}")
    if n_variants > 1:
        print(f"補正 : Bonferroni（{n_variants}バリアント） α_adj = {alpha_adj}")
    print(f"-" * 50)
    print(f"α = {alpha_adj} (z_α = {z_alpha:.4f})  |  Power = {power:.0%} (z_β = {z_beta:.4f})")
    print(f"N = [z_α√(p₀(1-p₀)) + z_β√(p̂(1-p̂))]² / (p̂ - p₀)²")
    print(f"  = [{z_alpha:.4f}×{np.sqrt(var_target):.6f} + {z_beta:.4f}×{np.sqrt(var_expected):.6f}]² / {effect_sq}")
    print(f"  = {required_n:,} visitors")
    print(f"-" * 50)

    # --- Plot ---
    offset_range = 0.10
    x_values = np.linspace(target_rate + 0.002, target_rate + offset_range, 500)
    y_values = (z_alpha * np.sqrt(var_target) + z_beta * np.sqrt(x_values * (1 - x_values))) ** 2 / ((x_values - target_rate) ** 2)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x_values, y=y_values, mode='lines', name='Required Traffic (CVR > target)', line=dict(color='#1f77b4', width=3)))

    if two_sided:
        x_below = np.linspace(max(0.001, target_rate - offset_range), target_rate - 0.002, 500)
        y_below = (z_alpha * np.sqrt(var_target) + z_beta * np.sqrt(x_below * (1 - x_below))) ** 2 / ((x_below - target_rate) ** 2)
        fig.add_trace(go.Scatter(x=x_below, y=y_below, mode='lines', name='Required Traffic (CVR < target)', line=dict(color='#ff7f0e', width=3, dash='dash')))

    fig.add_trace(go.Scatter(x=[expected_rate], y=[required_n], mode='markers', name='Your Plan',
        marker=dict(color='red', size=12, symbol='diamond')))
    fig.add_vline(x=target_rate, line_dash="dot", line_color="gray", annotation_text=f"Target: {target_rate:.1%}", annotation_position="top left")
    fig.add_annotation(x=expected_rate, y=required_n, text=f"<b>{required_n:,} Visitors</b><br>at {expected_rate:.1%}",
        ax=40, ay=-40, font=dict(size=12, color="red"), arrowcolor="red")

    bonf = f"  |  Bonferroni: α_adj = {alpha_adj}" if n_variants > 1 else ""
    fig.update_layout(
        title=dict(text=f"<b>Min Sample Size Planner ({test_label})</b><br>Target: {target_rate:.1%} | z_α = {z_alpha:.2f} | Power = {power:.0%}{bonf}", font=dict(size=20)),
        xaxis_title="True Conversion Rate", yaxis_title="Visitors Needed (N)",
        xaxis=dict(tickformat=".1%", showgrid=True, gridcolor='#eee'), yaxis=dict(showgrid=True, gridcolor='#eee'),
        template="plotly_white", plot_bgcolor='rgba(0,0,0,0)')
    fig.show()
    return required_n

In [ ]:
from scipy.optimize import brentq

def fleiss_mde(n, p0, z_alpha, z_beta):
    """Solve Fleiss one-sample N equation for delta (exact inverse of sample-size formula).
    Solves in both directions and returns the min (matches the design's direction)."""
    def _solve(sign):
        def residual(delta):
            p1 = p0 + sign * delta
            return (z_alpha * np.sqrt(p0 * (1 - p0)) + z_beta * np.sqrt(p1 * (1 - p1))) ** 2 / delta ** 2 - n
        delta_max = (1 - p0 if sign > 0 else p0) - 1e-10
        return brentq(residual, 1e-8, delta_max)
    return min(_solve(+1), _solve(-1))

def calculate_mde(planned_n, target_rate, two_sided=True, alpha=0.05, power=0.80, n_variants=1):
    alpha_adj = alpha / n_variants
    z_alpha = norm.ppf(1 - alpha_adj / 2) if two_sided else norm.ppf(1 - alpha_adj)
    z_beta = norm.ppf(power)
    test_label = "Two-Sided" if two_sided else "One-Sided"

    mde = fleiss_mde(planned_n, target_rate, z_alpha, z_beta)

    print(f"--- MINIMUM DETECTABLE EFFECT ({test_label}) ---")
    print(f"目標 : {target_rate:.1%}  |  訪問者数 : {planned_n:,}")
    if n_variants > 1:
        print(f"補正 : Bonferroni（{n_variants}バリアント） α_adj = {alpha_adj}")
    print(f"-" * 50)
    print(f"MDE = {mde:.2%}")
    print(f"検出可能範囲: {target_rate - mde:.2%} ~ {target_rate + mde:.2%}")
    print(f"-" * 50)

    # --- Plot ---
    n_range = np.linspace(max(100, planned_n * 0.1), planned_n * 3, 500)
    mde_curve = np.array([fleiss_mde(n, target_rate, z_alpha, z_beta) for n in n_range])

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=n_range, y=mde_curve, mode='lines', name='MDE vs N', line=dict(color='#1f77b4', width=3)))
    fig.add_trace(go.Scatter(x=[planned_n], y=[mde], mode='markers', name=f'Your Plan (N={planned_n:,})',
        marker=dict(color='red', size=12, symbol='diamond')))
    fig.add_annotation(x=planned_n, y=mde, text=f"<b>MDE = {mde:.2%}</b><br>at N = {planned_n:,}",
        ax=40, ay=-40, font=dict(size=12, color="red"), arrowcolor="red")

    bonf = f"  |  Bonferroni: α_adj = {alpha_adj}" if n_variants > 1 else ""
    fig.update_layout(
        title=dict(text=f"<b>MDE vs Sample Size ({test_label})</b><br>Target: {target_rate:.1%} | Power = {power:.0%}{bonf}", font=dict(size=20)),
        xaxis_title="Sample Size (N)", yaxis_title="MDE",
        xaxis=dict(showgrid=True, gridcolor='#eee'), yaxis=dict(tickformat=".2%", showgrid=True, gridcolor='#eee'),
        template="plotly_white", plot_bgcolor='rgba(0,0,0,0)')
    fig.show()
    return mde

In [31]:
planned_n = calculate_min_sample_size(
    target_rate=d["target_rate"], expected_rate=d["expected_rate"],
    two_sided=d["two_sided"], alpha=d["alpha"],
    power=d["power"], n_variants=d["n_variants"])

print()
calculate_mde(planned_n=planned_n, target_rate=d["target_rate"],
    two_sided=d["two_sided"], alpha=d["alpha"],
    power=d["power"], n_variants=d["n_variants"])

# Save computed results
config["computed"]["planned_n"] = planned_n
config["computed"]["alpha_adjusted"] = d["alpha"] / d["n_variants"]
json.dump(config, open("../test_config.json", "w"), indent=2)

--- SAMPLE SIZE PLANNER (Two-Sided) ---
手法 : 頻度論 z検定（単一比率の検定）
目標 : 2.0%  |  期待CVR : 1.0%
--------------------------------------------------
α = 0.05 (z_α = 1.9600)  |  Power = 80% (z_β = 0.8416)
N = [z_α√(p₀(1-p₀)) + z_β√(p̂(1-p̂))]² / (p̂ - p₀)²
  = [1.9600×0.140000 + 0.8416×0.099499]² / 0.0001
  = 1,283 visitors
--------------------------------------------------



--- MINIMUM DETECTABLE EFFECT (Two-Sided) ---
目標 : 2.0%  |  訪問者数 : 1,283
--------------------------------------------------
MDE = 1.10%
検出可能範囲: 0.90% ~ 3.10%
--------------------------------------------------
